# Lesson 05 - Agentic RAG

## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint, for example `https://azure-search-service-01.search.windows.net`
- `AZURE_SEARCH_INDEX_NAME` — your Azure AI Search index name, for example `demo-datasource-ks-index`
- `AZURE_SEARCH_API_KEY` — your Azure AI Search query or admin API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)

In [1]:
%pip install agent-framework azure-ai-projects azure-identity azure-search-documents python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
search_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
search_index_name = os.getenv("AZURE_SEARCH_INDEX_NAME", "demo-datasource-ks-index")
search_api_key = os.getenv("AZURE_SEARCH_API_KEY")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name,
    "AZURE_SEARCH_SERVICE_ENDPOINT": search_endpoint,
    "AZURE_SEARCH_API_KEY": search_api_key,
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [13]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

In [14]:
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=search_index_name,
    credential=AzureKeyCredential(search_api_key),
)

print(f"Azure AI Search index configured: {search_index_name}")

Azure AI Search index configured: demo-datasource-ks-index


In [15]:
search_client

<SearchClient [endpoint='https://azure-search-service-01.search.windows.net', index='demo-datasource-ks-index']>

## What is Agentic RAG?

Traditional RAG follows a fixed pipeline: retrieve documents, then generate a response. **Agentic RAG** goes further by giving the agent autonomy to decide **when** and **how** to retrieve information.

With Agentic RAG, the agent can:
- **Decide** whether retrieval is needed before answering a question
- **Choose** which data source or tool to query
- **Evaluate** retrieved results and perform follow-up retrievals if the first attempt is insufficient
- **Combine** information from multiple retrieval steps into a coherent answer

This makes the agent more flexible and accurate compared to a static retrieve-then-generate pipeline.

## Creating a Search Tool

In Agentic RAG, external data sources are wrapped as **tools** that the agent can invoke on demand. This lets the agent treat retrieval as just another action it can take, rather than a mandatory step.

Below we expose your Azure AI Search index as a tool the agent can call to look up destination information.

In [16]:
def format_search_result(result: dict) -> str:
    visible_fields = []

    for field_name, field_value in result.items():
        if field_name.startswith("@search."):
            continue
        if field_value is None:
            continue
        if field_name.lower().endswith("vector"):
            continue

        field_text = " ".join(str(field_value).split())
        if len(field_text) > 500:
            field_text = f"{field_text[:500]}..."
        visible_fields.append(f"{field_name}: {field_text}")

    return "\n".join(visible_fields)

# the search is a tool in Agentic RAG
@tool(approval_mode="never_require")
def search_knowledge_base(
    query: Annotated[str, "The search query for the Azure AI Search index"]
) -> str:
    """Search the Azure AI Search index for relevant information."""
    results = search_client.search(search_text=query, top=3)
    matches = []

    for result in results:
        formatted_result = format_search_result(dict(result))
        if formatted_result:
            matches.append(formatted_result)

    return (
        "\n\n---\n\n".join(matches)
        if matches
        else "No matching documents found in the Azure AI Search index."
    )

In [17]:
sample_results = search_client.search(search_text="batman", top=1)

for result in sample_results:
    print(format_search_result(dict(result)))

snippet_parent_id: aHR0cHM6Ly9zdGFjdGRlbW8yMDI2MDYyNS5ibG9iLmNvcmUud2luZG93cy5uZXQvZGVtby1kYXRhc291cmNlL3RoZS1kYXJrLWtuaWdodC0yMDA4LnBkZg2
blob_url: https://stactdemo20260625.blob.core.windows.net/demo-datasource/the-dark-knight-2008.pdf
uid: ac14eb9d3111_aHR0cHM6Ly9zdGFjdGRlbW8yMDI2MDYyNS5ibG9iLmNvcmUud2luZG93cy5uZXQvZGVtby1kYXRhc291cmNlL3RoZS1kYXJrLWtuaWdodC0yMDA4LnBkZg2_pages_80
snippet: image all of Gotham. (turns to Batman) This is wrong. BATMAN I've got to find this man, Lucius. FOX But at what cost? BATMAN The database is null-key encrypted. It can only be accessed by one person. FOX No one should have that kind of power. WAYNE That's why I gave it to you. Only you can use it. Lucius looks at Batman. Hard. FOX Spying on thirty million people wasn't in my job description. Batman points to a TV screen. Fox turns. ON SCREEN: the Joker shakes his head above a graphic "LATEST THR...


## Building the RAG Agent

Now we create an agent that is instructed to **always retrieve information before answering**. The agent uses the `search_knowledge_base` tool to ground its responses in your Azure AI Search index rather than relying on its own training data.

In [ ]:
agent = client.as_agent(
    tools=[search_knowledge_base],
    name="AzureSearchRAGAgent",
    instructions="""You are a knowledgeable assistant. Before answering questions:
1. ALWAYS search the Azure AI Search index first
2. Base your answers on retrieved information
3. If information is not in the index, say so clearly
4. Cite concrete details from the retrieved snippets.""",
)

response = await agent.run(
    "What does the knowledge base say about Barbie Land ?",
)
print(response)

The knowledge base does not contain any information about "Barbieland." If you have any other questions or need information on a different topic, feel free to ask!


## Iterative Retrieval — The Maker-Checker Pattern

A key advantage of Agentic RAG is **iterative retrieval**. The agent can perform multiple rounds of search to verify, refine, or expand on its initial findings — similar to a "maker-checker" workflow:

1. **Maker step**: The agent retrieves initial information and drafts a response.
2. **Checker step**: The agent performs additional retrievals to verify details or fill gaps.

Below, the agent is asked a question that requires comparing information from multiple retrieval results, prompting it to search several times.

In [11]:
checker_agent = client.as_agent(
    tools=[search_knowledge_base],
    name="AzureSearchRAGCheckerAgent",
    instructions="""You are a meticulous screenplay knowledge-base assistant who verifies answers against retrieved document snippets.
When answering questions about characters, scenes, dialogue, or events:
1. Search the Azure AI Search index for the main names, titles, or phrases in the question
2. Search again using specific character names, scene details, or quoted phrases found in the first results
3. Compare the retrieved snippets for consistent evidence about actions, relationships, and context
4. Ground the final answer in the retrieved snippets and mention the source document or blob URL when available
5. If the index does not contain enough evidence, clearly say what is missing instead of guessing.""",
)

response = await checker_agent.run(
    "Compare what the knowledge base says about Barbie and Ken.",
)
print(response)

Based on the retrieved screenplay snippets from the "BARBIE FINAL 2023" script:

Barbie (referred to as Barbie Margot) is shown to experience complex emotions such as fear and anxiety ("I’ve started to get all these weirdo FEELINGS," "like I have fear with no specific object"). She interacts with Weird Barbie and shares moments of vulnerability ("My feet -- they’re..."). Barbie seems to be on a quest or mission, trying to find someone ("We better find her soon," "She’s got to be here somewhere..."). She also goes through emotional lows ("This is the lowest I’ve ever been. Emotionally AND physically.").

Ken (referred to as Ken Ryan Gosling) is portrayed with a focus on masculine identity and activities ("Every Ken is there... now bearded Ken Ryan Gosling sporting a mink coat," "rotating through all the man-activities"). He appears confident and upbeat ("I feel amazing," "Cool! Kids are running everywhere"). Ken has a kind of group presence with other Kens and speaks about realizations 

## Summary

In this lesson you learned how to build an **Agentic RAG** system using the Microsoft Agent Framework:

- **Agentic RAG** lets agents autonomously decide when to retrieve information, making retrieval dynamic rather than fixed.
- **Tools as data sources**: External knowledge bases, like Azure AI Search indexes, are wrapped as tools the agent can invoke.
- **Iterative retrieval**: The maker-checker pattern enables the agent to perform multiple retrieval rounds — searching, verifying, and refining — before producing a final answer.

This pattern lets you replace static in-memory data with a real Azure AI Search index while keeping retrieval under the agent's control.